# Content Attribution Evaluation Tutorial

This notebook demonstrates how to evaluate **content attribution** — measuring what percentage of a report's content comes from:
- **Web sources** (external real-time data from Tavily)
- **Internal sources** (RAG documents, company knowledge bases)
- **Prior knowledge** (model's training data)

**How it works:** The report is split into sentences, then an LLM judge classifies each sentence as originating from a web source, internal source, or prior knowledge. This gives sentence-level granularity with the semantic understanding of an LLM.

**Why this matters:**
- Quantify the value of web data in your research outputs
- Measure contribution of internal knowledge bases in enterprise RAG systems
- Understand how much the model relies on its training data vs. fresh sources
- Optimize your retrieval strategy based on content contribution

## Setup

In [2]:
%pip install -q tavily-agent-toolkit


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("TAVILY_API_KEY:\n")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY:\n")

In [6]:
from tavily_agent_toolkit import ModelConfig, ModelObject, search_and_answer
from tavily_agent_toolkit.evals import compute_content_attribution_metrics

judge_config = ModelConfig(
    model=ModelObject(
        model="gpt-5-mini",
        api_key=os.environ["OPENAI_API_KEY"],
    ),
)

## Example 1: High Web Content (Fresh Data)

A report about recent events should have high web content ratio.

In [7]:
# Report based heavily on recent web sources
report_fresh_data = """
According to NVIDIA's latest earnings report, the company achieved record revenue of $35.1 billion 
in Q3 2024, a 94% increase year-over-year. Data Center revenue reached $30.8 billion.

The company's stock price rose 15% following the announcement, with analysts raising price 
targets across the board. Goldman Sachs increased their target to $800, citing strong AI demand.
"""

sources_fresh = [
    {
        "url": "https://investor.nvidia.com/q3-2024",
        "title": "NVIDIA Q3 2024 Earnings",
        "content": "NVIDIA announced quarterly revenue of $35.1 billion for Q3 fiscal 2024, up 94% from a year ago. Data Center revenue was a record $30.8 billion."
    },
    {
        "url": "https://finance.yahoo.com/nvidia-earnings-reaction",
        "title": "NVIDIA Stock Surges on Earnings",
        "content": "NVIDIA shares jumped 15% after reporting blowout earnings. Goldman Sachs raised their price target to $800, with multiple analysts citing sustained AI chip demand."
    }
]

In [8]:
result_fresh = await compute_content_attribution_metrics(
    report=report_fresh_data,
    web_sources=sources_fresh,
    judge_model_config=judge_config,
)

print("Content Attribution Breakdown:")
print(f"  Web Content:      {result_fresh.web_content_ratio:.1%}")
print(f"  Internal Content: {result_fresh.internal_content_ratio:.1%}")
print(f"  Prior Knowledge:  {result_fresh.prior_knowledge_ratio:.1%}")
print(f"\nSource Diversity: {result_fresh.source_diversity:.1%}")
print(f"Unique Domains: {result_fresh.unique_domains}")

Content Attribution Breakdown:
  Web Content:      100.0%
  Internal Content: 0.0%
  Prior Knowledge:  0.0%

Source Diversity: 100.0%
Unique Domains: 2


## Example 2: High Prior Knowledge (General Topics)

A report about general concepts will rely more on model prior knowledge.

In [9]:
# Report with general background mixed with specific data
report_general = """
Artificial intelligence (AI) refers to computer systems designed to perform tasks that 
typically require human intelligence. Machine learning, a subset of AI, enables systems 
to learn from data without explicit programming.

NVIDIA is a leading provider of GPUs for AI training. Their H100 chip is widely used 
by companies like OpenAI and Google for training large language models.
"""

sources_general = [
    {
        "url": "https://techcrunch.com/nvidia-ai",
        "title": "NVIDIA AI Chips",
        "content": "NVIDIA's H100 GPU has become the gold standard for AI training, with adoption by OpenAI, Google, and other major AI labs."
    }
]

result_general = await compute_content_attribution_metrics(
    report=report_general,
    web_sources=sources_general,
    judge_model_config=judge_config,
)

print("Content Attribution Breakdown:")
print(f"  Web Content:      {result_general.web_content_ratio:.1%}")
print(f"  Internal Content: {result_general.internal_content_ratio:.1%}")
print(f"  Prior Knowledge:  {result_general.prior_knowledge_ratio:.1%}")

Content Attribution Breakdown:
  Web Content:      41.1%
  Internal Content: 0.0%
  Prior Knowledge:  58.9%


## Example 3: Hybrid Sources (Web + Internal)

Most enterprise use cases combine external web data with internal knowledge bases.

In [10]:
# Report combining external market data with internal context
report_hybrid = """
The cloud infrastructure market grew 22% YoY in 2024, reaching $270 billion globally 
according to recent industry reports. AWS maintained market leadership at 32% share.

Our company's cloud spending increased 18% this year to $2.4M annually. Based on our 
infrastructure team's analysis, migrating to a multi-cloud strategy could reduce costs 
by approximately 15% while improving redundancy.
"""

# External web sources
web_sources_hybrid = [
    {
        "url": "https://cloudreport.com/2024-market",
        "title": "Cloud Infrastructure Market Report 2024",
        "content": "Global cloud infrastructure market reached $270B in 2024, up 22% YoY. AWS leads with 32% market share, followed by Azure at 23% and GCP at 11%."
    }
]

# Internal company knowledge (RAG documents)
internal_docs = [
    {
        "title": "IT Budget Analysis 2024",
        "content": """Document ID: FIN-2024-0892
Department: Finance / IT Operations
Last Updated: December 2024

CLOUD INFRASTRUCTURE SPENDING SUMMARY

Total annual cloud spending: $2.4M (up 18% from prior year)
Primary providers breakdown:
- AWS: 60% ($1.44M)
- Azure: 30% ($720K)  
- GCP: 10% ($240K)

Budget approved for FY2025: $2.8M (+17% YoY)"""
    },
    {
        "title": "Multi-Cloud Strategy Proposal",
        "content": """Document ID: TECH-2024-0156
Author: Infrastructure Team
Status: Under Review

EXECUTIVE SUMMARY

The infrastructure team recommends adopting a multi-cloud strategy 
to reduce vendor lock-in and improve system redundancy.

Key findings:
- Projected cost savings: 15% annually
- Improved disaster recovery capabilities
- Better negotiating leverage with vendors
- Estimated implementation timeline: 6-9 months"""
    }
]

result_hybrid = await compute_content_attribution_metrics(
    report=report_hybrid,
    web_sources=web_sources_hybrid,
    internal_sources=internal_docs,
    judge_model_config=judge_config,
)

print("Content Attribution Breakdown:")
print(f"  Web Content:      {result_hybrid.web_content_ratio:.1%}")
print(f"  Internal Content: {result_hybrid.internal_content_ratio:.1%}")
print(f"  Prior Knowledge:  {result_hybrid.prior_knowledge_ratio:.1%}")
print(f"\nTotal Sources: {result_hybrid.total_sources}")

Content Attribution Breakdown:
  Web Content:      43.3%
  Internal Content: 56.7%
  Prior Knowledge:  0.0%

Total Sources: 3


## Example 4: Real-World Analysis

Let's run a real search and analyze the content attribution.

In [11]:
# Search for recent, time-sensitive information
search_result = await search_and_answer(
    query="What are the latest developments in AI regulation in 2025?",
    api_key=os.environ["TAVILY_API_KEY"],
    model_config=judge_config,
    max_results=5,
)

print("Query: What are the latest developments in AI regulation in 2025?")
print(f"Sources retrieved: {len(search_result['results'])}")

Query: What are the latest developments in AI regulation in 2025?
Sources retrieved: 5


In [9]:
# Analyze content attribution
attribution = await compute_content_attribution_metrics(
    report=search_result["answer"],
    web_sources=search_result["results"],
    judge_model_config=judge_config,
)

print("\nContent Attribution Analysis:")
print("="*50)
print(f"Web Sources:      {attribution.web_content_ratio:.1%}  <- Value from Tavily")
print(f"Prior Knowledge:  {attribution.prior_knowledge_ratio:.1%}  <- Model's training data")
print(f"\nSource Diversity: {attribution.source_diversity:.1%}")
print(f"Unique Domains:   {attribution.unique_domains}")
print("="*50)


Content Attribution Analysis:
Web Sources:      95.0%  <- Value from Tavily
Prior Knowledge:  5.0%  <- Model's training data

Source Diversity: 100.0%
Unique Domains:   5


In [13]:
print("LLM Answer:")
print("-" * 50)
print(search_result["answer"])
print()
print("Tavily Sources:")
print("-" * 50)
for i, src in enumerate(search_result["results"], 1):
    print(f"  [{i}] {src['title']}")
    print(f"      {src['url']}")
    print()

LLM Answer:
--------------------------------------------------
Major 2025 developments in AI regulation:

- EU: Key provisions of the EU AI Act began to take effect in 2025, marking a major regulatory milestone for high‑risk AI systems and prompting enterprises to prioritize governance, risk assessment, documentation and controls to achieve compliance (Credo AI) (FPF).  

- China: The Cyberspace Administration of China (CAC) issued the final "Measures for Labeling AI‑Generated Content" in March 2025, requiring online services that create or distribute AI‑generated content to clearly label it (effective September 1, 2025). China’s standards bodies have also produced technical drafts (e.g., NISSTC’s 2024 draft Security Requirements for Generative AI) tightening requirements for training data and model security (Cimplifi).  

- United Kingdom: The UK has emphasized a sectoral, principles‑based approach (safety, transparency, fairness, accountability, contestability) rather than a single A

## Use Cases

1. **Quantify Web Data Value**: Show stakeholders how much value search adds
2. **Optimize Retrieval**: If prior knowledge is too high, retrieve more sources
3. **Quality Assurance**: Ensure reports are grounded in fresh data
4. **Cost Analysis**: Correlate web content ratio with search credits spent
5. **Enterprise RAG Value**: Measure contribution of internal knowledge bases
6. **Knowledge Gap Analysis**: Identify topics where internal data is missing